# FunSearch — Islands

Adding island-based populations to prevent premature convergence. Multiple separate DBs evolve independently, migration shares best ideas across islands.

## 1. LLM Setup

In [1]:
import os
import random
from litellm import completion
import litellm
from statistics import mean
from dataclasses import dataclass

model_name = "nvidia_nim/openai/gpt-oss-20b"
litellm.register_model({model_name: {
    "max_tokens": 8192,
    "input_cost_per_token": 0.0,
    "output_cost_per_token": 0.0,
    "supports_assistant_prefill": False,
}})

{'nvidia_nim/openai/gpt-oss-20b': {'max_tokens': 8192,
  'input_cost_per_token': 0.0,
  'output_cost_per_token': 0.0,
  'supports_assistant_prefill': False}}

## 2. Sampler

In [2]:
def sample_code(prompt: str, temp:float = 0.8) -> str:
    resp = completion(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
    )
    text = resp.choices[0].message.content
    if "```" in text:
        code = text.split("```")[1]
        if code.startswith("python"):
            code = code[len("python"):]
        return code.strip()
    return text.strip()

## 3. DB entry + Prompt Builder

In [3]:
from IPython.display import display, Markdown


@dataclass
class DB:
    code : str
    score : float

    def __str__(self):
        preview = self.code.strip().split('\n')[0]
        return f"[score={self.score:.4f}] {preview}"


    def display_code(self):
        display(Markdown(f"**Score: {self.score:.4f}**\n```python\n{self.code}\n```"))
def build_prompt(spec, best_programs):
    examples = "\n\n".join(
        f"# score: {p.score}\n{p.code}" for p in best_programs
    )
    return f"""Here's the task: {spec}

Here are the best solutions so far:
{examples}

Write an improved version. Return ONLY the function, no explanation."""

## 4. Evaluator (string compression)

In [4]:
test_strings = [
    "aaaaaabbbbbccccdddeef",
    "the cat sat on the mat and the cat saw the bat",
    "abcabcabcabcabcabc",
    "aaaaaaaaaaaaaaaaaaaabbbbbbbbbbbbbbbbcccccccccc",
    "xyxyxyxyxyxyxyxyxyxy",
    "hello world hello world hello world",
    "abcdefghijklmnopqrstuvwxyz",
    "aaabbbcccaaabbbcccaaabbbccc",
]

def evaluate_compression(code_str):
    namespace = {}
    try:
        exec(code_str, namespace)
    except Exception:
        return 0.0

    if "compress" not in namespace or "decompress" not in namespace:
        return 0.0

    ratios = []
    for s in test_strings:
        try:
            compressed = namespace["compress"](s)
            decompressed = namespace["decompress"](compressed)
            if decompressed != s:
                ratios.append(0.0)
            elif len(compressed) >= len(s):
                ratios.append(0.0)
            else:
                ratios.append(1.0 - len(compressed) / len(s))
        except Exception:
            ratios.append(0.0)

    return sum(ratios) / len(ratios)

## 5. Seed + Spec

In [5]:
spec = """Write two functions: `compress(s)` and `decompress(s)`.
- compress takes a string and returns a shorter string encoding
- decompress takes the compressed string and returns the original
- decompress(compress(s)) must equal s exactly
- goal: make the compressed output as short as possible
- do NOT use any imports or standard library compression"""

seed = '''def compress(s):
    out = []
    i = 0
    while i < len(s):
        c = s[i]
        count = 1
        while i + count < len(s) and s[i + count] == c:
            count += 1
        if count > 1:
            out.append(f"{count}{c}")
        else:
            out.append(c)
        i += count
    return "".join(out)

def decompress(s):
    out = []
    i = 0
    while i < len(s):
        if s[i].isdigit():
            num = []
            while i < len(s) and s[i].isdigit():
                num.append(s[i])
                i += 1
            out.append(int("".join(num)) * s[i])
            i += 1
        else:
            out.append(s[i])
            i += 1
    return "".join(out)'''

## 6. Islands — YOUR PART: write `db_add`, `best`, migration, and the loop
Replace single `lis` with multiple islands. Each iteration: pick random island → best(2) from it → prompt → evaluate → store in that island. Every N iters, migrate best from one island to another.

In [6]:
from random import randint

class DB_list:
    def __init__(self, count=2, N=20):
        self.count = count 
        self.lis = [[] for _ in range(count)]
        self.N = 20 

    def rand_idx(self):
        return randint(0, self.count-1)
    
    def db_add(self, code, score, idx=None):
        if idx is None:
            idx = self.rand_idx()

        self.lis[idx].append(DB(code=code, score=score))
        self.lis[idx].sort(key=lambda x: x.score, reverse=True)
        self.lis[idx] = self.lis[idx][:self.N]

    def best(self, k, idx=None): 

        if idx is None:
            idx = self.rand_idx()

        return self.lis[idx][:k], idx

    def migrate(self, src, dst):
        programs, _ = self.best(1, idx=src)
        if programs:
            self.db_add(programs[0].code, score=programs[0].score, idx=dst)
    
    def cull_worst_island(self, seed_code, seed_score):
        worst = min(range(self.count), key=lambda i: self.lis[i][0].score if self.lis[i] else float("-inf"))
        self.lis[worst] = [DB(code=seed_code, score=seed_score)]
        print(f"Culled island {worst}")

In [7]:
seed

'def compress(s):\n    out = []\n    i = 0\n    while i < len(s):\n        c = s[i]\n        count = 1\n        while i + count < len(s) and s[i + count] == c:\n            count += 1\n        if count > 1:\n            out.append(f"{count}{c}")\n        else:\n            out.append(c)\n        i += count\n    return "".join(out)\n\ndef decompress(s):\n    out = []\n    i = 0\n    while i < len(s):\n        if s[i].isdigit():\n            num = []\n            while i < len(s) and s[i].isdigit():\n                num.append(s[i])\n                i += 1\n            out.append(int("".join(num)) * s[i])\n            i += 1\n        else:\n            out.append(s[i])\n            i += 1\n    return "".join(out)'

In [8]:
lis = DB_list(3)
seed_score = evaluate_compression(seed)
for i in range(lis.count):
    lis.db_add(seed, seed_score, idx=i)
print(f"Seeded {lis.count} islands with score {seed_score:.4f}")

Seeded 3 islands with score 0.2017


In [9]:
temps = {0: 0.5, 1: 0.8, 2: 1.2} # as there are 3 islands

In [10]:
from concurrent.futures import ThreadPoolExecutor

def sample_multiple(prompt, n=3, temp=1.2):

    with ThreadPoolExecutor(max_workers=n) as pool:
        futures = [pool.submit(sample_code, prompt, temp) for _ in range(n)]
        results = [f.result() for f in futures]
        
    return results 

In [11]:
sample_multiple("write a function to add two numbers.", n = 2 )

['def add(a, b):\n    """Return the sum of a and b."""\n    return a + b\n\n\n# Example usage\nprint(add(3, 5))  # → 8',
 'def add(a: float, b: float) -> float:\n    """\n    Add two numbers and return the sum.\n    \n    Parameters\n    ----------\n    a : float\n        First addend\n    b : float\n        Second addend\n    \n    Returns\n    -------\n    float\n        The sum of a and b\n    """\n    return a + b\n\n# Demo\nprint(add(3, 4))          # 7\nprint(add(-1.5, 2.5))     # 1.0']

In [12]:
import random

for i in range(100):
    best_programs, idx = lis.best(2)

    prompt = build_prompt(spec, best_programs)

    #new_code = sample_code(prompt)

    for new_code in sample_multiple(prompt, n=3, temp=temps[idx]):
        score = evaluate_compression(new_code)
        lis.db_add(new_code, score, idx=idx)

    if score == 0.0:
        # debug: why did it fail?
        ns = {}
        try:
            exec(new_code, ns)
            if "compress" in ns and "decompress" in ns:
                t = test_strings[0]
                c = ns["compress"](t)
                d = ns["decompress"](c)
                print(f"  iter {i}: score=0 | orig={t!r} comp={c!r} decomp={d!r} match={d==t}")
            else:
                print(f"  iter {i}: score=0 | missing compress/decompress")
        except Exception as e:
            print(f"  iter {i}: score=0 | exec error: {e}")
    else:
        island_best = lis.best(1, idx=idx)[0][0].score
        print(f"  iter {i}: island={idx} score={score:.4f}  island_best={island_best:.4f}")
    
    if i % 10==0:
        src, dst = random.sample(range(lis.count), 2)
        lis.migrate(src,dst)

    if i % 30 == 0 :
        seed_score = evaluate_compression(seed)
        lis.cull_worst_island(seed, seed_score)

  iter 0: island=0 score=0.2017  island_best=0.2017
Culled island 0
  iter 1: island=2 score=0.2017  island_best=0.2017
  iter 2: island=0 score=0.4499  island_best=0.4499
  iter 3: island=1 score=0.0842  island_best=0.2626
  iter 4: island=0 score=0.3862  island_best=0.4499
  iter 5: island=0 score=0.4499  island_best=0.4499
  iter 6: island=0 score=0.3734  island_best=0.4760
  iter 7: island=2 score=0.2099  island_best=0.2099
  iter 8: island=1 score=0.3656  island_best=0.3862
  iter 9: island=2 score=0.1281  island_best=0.2099
  iter 10: island=0 score=0.3513  island_best=0.4760
  iter 11: island=2 score=0.2099  island_best=0.2099
  iter 12: island=0 score=0.3854  island_best=0.4760
  iter 13: score=0 | orig='aaaaaabbbbbccccdddeef' comp='\x01\x01a\x06\x01\x01b\x05\x01\x01c\x04dddeef' decomp='\x01a\x06\x01b\x05\x01c\x04dddeef' match=False
  iter 14: island=1 score=0.0217  island_best=0.4063
  iter 15: island=1 score=0.0924  island_best=0.4063
  iter 16: island=1 score=0.1097  island_

In [13]:
for i in range(lis.count):
    progs, _ = lis.best(1, idx=i)
    print(f"Island {i}: best={progs[0].score:.4f}" if progs else f"Island {i}: empty")

Island 0: best=0.4814
Island 1: best=0.4814
Island 2: best=0.2017


In [15]:
lis.lis[0][0].display_code()

**Score: 0.4814**
```python
def compress(s):
    ESC = '\x01'
    REF = '\x00'
    RLE = '\x02'
    markers = {ESC, REF, RLE}
    n = len(s)
    dp = [0] * (n + 1)
    back = [None] * (n + 1)
    dp[n] = 0
    for i in range(n - 1, -1, -1):
        # literal
        if s[i] in markers:
            best_cost = 2 + dp[i + 1]
            best_action = (0, None)
        else:
            best_cost = 1 + dp[i + 1]
            best_action = (0, None)
        # run
        run_len = 1
        while i + run_len < n and s[i + run_len] == s[i] and run_len < 255:
            run_len += 1
        if run_len >= 3:
            cost = 3 + dp[i + run_len]
            if cost < best_cost:
                best_cost = cost
                best_action = (1, run_len)
        # reference
        best_len = 0
        best_off = 0
        start = max(0, i - 255)
        for j in range(start, i):
            l = 0
            while i + l < n and s[j + l] == s[i + l] and l < 255:
                l += 1
            if l > best_len:
                best_len = l
                best_off = i - j
                if best_len == 255:
                    break
        if best_len >= 3:
            cost = 3 + dp[i + best_len]
            if cost < best_cost:
                best_cost = cost
                best_action = (2, (best_off, best_len))
        dp[i] = best_cost
        back[i] = best_action
    # reconstruct
    out = []
    i = 0
    while i < n:
        action, param = back[i]
        if action == 0:
            c = s[i]
            if c in markers:
                out.append(ESC)
                out.append(c)
            else:
                out.append(c)
            i += 1
        elif action == 1:
            run_len = param
            out.append(RLE)
            out.append(s[i])
            out.append(chr(run_len))
            i += run_len
        else:
            offset, length = param
            out.append(REF)
            out.append(chr(offset))
            out.append(chr(length))
            i += length
    return ''.join(out)


def decompress(s):
    ESC = '\x01'
    REF = '\x00'
    RLE = '\x02'
    out = []
    i = 0
    n = len(s)
    while i < n:
        c = s[i]
        if c == ESC:
            i += 1
            out.append(s[i])
            i += 1
        elif c == REF:
            offset = ord(s[i + 1])
            length = ord(s[i + 2])
            start = len(out) - offset
            for k in range(length):
                out.append(out[start + k])
            i += 3
        elif c == RLE:
            ch = s[i + 1]
            cnt = ord(s[i + 2])
            out.extend([ch] * cnt)
            i += 3
        else:
            out.append(c)
            i += 1
    return ''.join(out)
```